## Problem Statement

*Exercise 2.12 from Operations Research: Models and Methods by Jensen & Bard*

Ten jobs are to be completed by three workers during the next week. Each worker has a 40-hour work week. The times for the workers to complete the jobs are shown in the table. The values in the cells assume that each job is completed by a single worker; however, jobs can be shared, with completion times being determined proportionally If no entry exists in a particular cell, it means that the corresponding job cannot be performed by the corresponding worker. Set up and solve an LP model that will determine the optimal assignment of workers to jobs. The goal is to minimize the total time required to complete all the jobs.

| Workers \ Tasks |  1 |  2 |  3 |  4 |  5 |  6 |  7 |  8 |  9 | 10 |
|:---------------:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| A               |  - |  7 |  3 |  - |  - | 18 | 13 |  6 |  - |  9 |
| B               | 12 |  5 |  - | 12 |  4 | 22 |  - | 17 | 13 |  - |
| C               | 18 |  - |  6 |  8 | 10 |  - | 19 |  - |  8 | 15 |

## Import

In [6]:
from collections import defaultdict

import pyomo.environ as pe
import pyomo.opt as po

## Define Data

In [2]:
workers = {'A', 'B', 'C'}

tasks = set(range(1, 11))

c = defaultdict(lambda: 1000, {
    ('A',  2):  7,
    ('A',  3):  3,
    ('A',  6): 18,
    ('A',  7): 13,
    ('A',  8):  6,
    ('A', 10):  9,
    ('B',  1): 12,
    ('B',  2):  5,
    ('B',  4): 12,
    ('B',  5):  4,
    ('B',  6): 22,
    ('B',  8): 17,
    ('B',  9): 13,
    ('C',  1): 18,
    ('C',  3):  6,
    ('C',  4):  8,
    ('C',  5): 10,
    ('C',  7): 19,
    ('C',  9):  8,
    ('C', 10): 15,
})

max_hours = 40

 ## Model
Define $W$ como el conjunto de trabajadores y $T$ como el conjunto de tareas.

Además, define $c_{wt}$ como el número de horas que el trabajador $w$ requiere para completar la tarea $t$.
(Observa que no prohibimos explícitamente a un trabajador completar una tarea; en su lugar, hacemos que el costo sea arbitrariamente grande si el trabajador $w$ no puede realizar la tarea $t$.)

Sea $x_{wt}$ la proporción de la tarea $t$ que es completada por el trabajador $j$.

Sea $H$ el número máximo de horas que cualquier trabajador individual puede registrar en una semana.

\textbf{Formulamos de la siguiente manera:}


$$
\begin{alignat*}{3}
\text{minimize  }  & \sum_{w \in W} \sum_{t \in T} c_{wt} x_{wt} && \\
\text{subject to  }
& \sum_{t \in T} c_{wt} x_{wt} \le H,
&& \qquad \forall w \in W \\
& \sum_{w \in W} x_{wt} = 1
&& \qquad \forall t \in T \\
& 0 \le x_{wt} \le 1,
&& \qquad \forall w \in W, \forall t \in T
\end{alignat*}
$$

## Implement

In [7]:
model = pe.ConcreteModel()

In [8]:
model.workers = pe.Set(initialize=workers)
model.tasks = pe.Set(initialize=tasks)

source (type: set).  This WILL potentially lead to nondeterministic behavior
in Pyomo
source (type: set).  This WILL potentially lead to nondeterministic behavior
in Pyomo


In [9]:
model.c = pe.Param(model.workers, model.tasks, initialize=c, default=1000)
model.max_hours = pe.Param(initialize=max_hours)
model.x = pe.Var(model.workers, model.tasks, domain=pe.Reals, bounds=(0, 1))

In [11]:
model.tasks_done = pe.ConstraintList()
for t in model.tasks:
    lhs = sum(model.x[w, t] for w in model.workers)
    rhs = 1
    model.tasks_done.add(lhs == rhs)

In [10]:
expr = sum(model.c[w, t] * model.x[w, t]
           for w in model.workers for t in model.tasks)
model.objective = pe.Objective(sense=pe.minimize, expr=expr)

In [12]:
model.hour_limit = pe.ConstraintList()
for w in model.workers:
    lhs = sum(model.c[w, t] * model.x[w, t] for t in model.tasks)
    rhs = model.max_hours
    model.hour_limit.add(lhs <= rhs)

## Solve and Postprocess

In [13]:
solver = po.SolverFactory('glpk')
results = solver.solve(model, tee=True)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpg2rz0tsy.glpk.raw
 --wglp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpuvxg167d.glpk.glp
 --cpxlp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpb2q9fpe3.pyomo.lp
Reading problem data from '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpb2q9fpe3.pyomo.lp'...
13 rows, 30 columns, 60 non-zeros
168 lines were read
Writing problem data to '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpuvxg167d.glpk.glp'...
179 lines were written
GLPK Simplex Optimizer 5.0
13 rows, 30 columns, 60 non-zeros
Preprocessing...
13 rows, 30 columns, 60 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+03  ratio =  1.000e+03
GM: min|aij| =  2.383e-01  max|aij| =  4.197e+00  ratio =  1.761e+01
EQ: min|aij| =  5.678e-02  max|aij| =  1.000e+00  ratio =  1.761e+01
Constructing initial basis...
Size of triangular part is 13
      0: obj =   4.056

In [14]:
import pandas as pd

df = pd.DataFrame(index=pd.MultiIndex.from_tuples(model.x, names=['w', 't']))
df['x'] = [pe.value(model.x[key]) for key in df.index]
df['c'] = [model.c[key] for key in df.index]

/Users/erickavendanogarcia/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [15]:
df

x     c
w t            
B 1   1.0    12
  2   1.0     5
  3   0.0  1000
  4   0.0    12
  5   1.0     4
  6   0.5    22
  7   0.0  1000
  8   0.0    17
  9   0.0    13
  10  0.0  1000
C 1   0.0    18
  2   0.0  1000
  3   0.0     6
  4   1.0     8
  5   0.0    10
  6   0.0  1000
  7   0.0    19
  8   0.0  1000
  9   1.0     8
  10  0.0    15
A 1   0.0  1000
  2   0.0     7
  3   1.0     3
  4   0.0  1000
  5   0.0  1000
  6   0.5    18
  7   1.0    13
  8   1.0     6
  9   0.0  1000
  10  1.0     9

In [16]:
(df['c'] * df['x']).unstack('t')

t,1,2,3,4,5,6,7,8,9,10
w,,,,,,,,,,
A,0.0,0.0,3.0,0.0,0.0,9.0,13.0,6.0,0.0,9.0
B,12.0,5.0,0.0,0.0,4.0,11.0,0.0,0.0,0.0,0.0
C,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,8.0,0.0


In [17]:
(df['c'] * df['x']).groupby('w').sum().to_frame()

,0
w,
A,40.0
B,32.0
C,16.0


In [18]:
df['x'].groupby('t').sum().to_frame().T

t,1,2,3,4,5,6,7,8,9,10
x,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
